# Weekly Laser and CBP Checkout (BLOCK-T683)

This notebook evaluates the outcome of the weekly calibration system test **BLOCK-T683**, which exercises the Tunable Laser and the Collimated Beam Projector (CBP).

The block performs the following steps:
1. **Laser functional test** – runs the `laser_functional` sequence, scanning wavelengths and recording electrometer signals
2. **CBP motion test** – exercises azimuth/elevation moves, focus changes, mask changes, and mask rotation changes
3. **Laser + CBP combined test** – runs the `laser_cbp` sequence with both systems active

**Expected CBP motion sequence:**
- Azimuth: unpark → 15.0° → -15.0° → park
- Elevation: unpark → 0.0° → -20.0° → park
- Focus: 5000 → 3800
- Mask: 5 → 1
- Mask rotation: 275.0° → 96.5°

In [ ]:
# User input
day_obs = 20250410

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.time import Time, TimeDelta
from lsst.resources import ResourcePath
from lsst_efd_client import EfdClient

In [ ]:
date_str = str(day_obs)
date_str_fmt = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}T23:59:00.00"
end_time = Time(date_str_fmt, format="isot")
start_time = end_time - TimeDelta(60.0 * 60 * 24, format="sec")

client = EfdClient("usdf_efd")
print(f"Querying EFD from {start_time.iso} to {end_time.iso}")

## 1. Electrometer Data

Electrometer signals from the `laser_functional` and `laser_cbp` sequences.
Images are identified by the block ID **BT683** in the electrometer large-file event IDs.
The earlier exposures (by timestamp) correspond to `laser_functional`; the later ones to `laser_cbp`.

In [ ]:
msg_log_topic = "lsst.sal.Electrometer.logevent_largeFileObjectAvailable"
elec_df = await client.select_time_series(
    msg_log_topic, ["url", "id", "salIndex"], start=start_time, end=end_time
)

if len(elec_df) > 0:
    block_df = elec_df[elec_df["id"].str.contains("BT683", case=False, na=False)].copy()
    block_df["dayobs"] = block_df["id"].str.split("_").str[2]
    day_block_df = block_df[block_df["dayobs"] == str(day_obs)].sort_index()
    print(f"Found {len(day_block_df)} electrometer exposures for BLOCK-T683 on {day_obs}")
else:
    day_block_df = pd.DataFrame()
    print(f"No electrometer data found on {day_obs}")

day_block_df

In [ ]:
if len(day_block_df) > 0:
    n = len(day_block_df)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axarr = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
    axes = axarr.flatten() if n > 1 else [axarr]

    for i, (idx, row) in enumerate(day_block_df.iterrows()):
        ax = axes[i]
        try:
            path = ResourcePath("s3://lfa@" + row.url.split(".org/")[1])
            with path.open("rb") as f:
                hdu = fits.open(f)
                signal = hdu[1].data["Signal"]
                time_arr = hdu[1].data["Elapsed Time"]
            mean_s, std_s = np.mean(signal), np.std(signal)
            ax.plot(time_arr, signal, "x-", label=f"Mean={mean_s:.3e}\nStd={std_s:.3e}")
            ax.set_title(f"Elec {row.salIndex}: {row['id']}", fontsize=8)
            ax.set_xlabel("Elapsed Time (s)")
            ax.set_ylabel("Current (A)")
            ax.legend(fontsize=7)
        except Exception as e:
            ax.set_title(f"Error: {row['id']}\n{e}", fontsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"BLOCK-T683 Electrometer Data – DayObs: {day_obs}", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No electrometer data to plot.")

## 2. FiberSpectrograph Spectra (laser_functional)

Spectra recorded during the `laser_functional` sequence. The laser steps through **8 wavelengths from 300 to 1000 nm** (center 700 nm ± 400 nm, step 100 nm). Each trace is labeled with its commanded laser wavelength. Spectra are grouped by FiberSpectrograph `salIndex`.

In [ ]:
fs_topic = "lsst.sal.FiberSpectrograph.logevent_largeFileObjectAvailable"
fs_df = await client.select_time_series(
    fs_topic, ["url", "id", "salIndex"], start=start_time, end=end_time
)

if len(fs_df) > 0:
    fs_block_df = fs_df[fs_df["id"].str.contains("BT683", case=False, na=False)].copy()
    fs_block_df["dayobs"] = fs_block_df["id"].str.split("_").str[2]
    day_fs_df = fs_block_df[fs_block_df["dayobs"] == str(day_obs)].sort_index()
    print(f"Found {len(day_fs_df)} FiberSpectrograph exposures for BLOCK-T683 on {day_obs}")
else:
    day_fs_df = pd.DataFrame()
    print(f"No FiberSpectrograph data found on {day_obs}")

day_fs_df

In [ ]:
# Plot FiberSpectrograph spectra grouped by instrument, one trace per laser wavelength step.
# laser_functional: center=700 nm, width=800 nm, step=100 nm → 8 steps: 300–1000 nm
laser_wavelengths = [300, 400, 500, 600, 700, 800, 900, 1000]

if len(day_fs_df) > 0:
    sal_indices = sorted(day_fs_df["salIndex"].unique())
    fig, axes = plt.subplots(1, len(sal_indices), figsize=(7 * len(sal_indices), 5), squeeze=False)
    axes = axes[0]

    for ax, sal_idx in zip(axes, sal_indices):
        sub_df = day_fs_df[day_fs_df["salIndex"] == sal_idx].sort_index()
        for i, (ts, row) in enumerate(sub_df.iterrows()):
            laser_wl = laser_wavelengths[i] if i < len(laser_wavelengths) else "?"
            try:
                path = ResourcePath("s3://lfa@" + row.url.split(".org/")[1])
                with path.open("rb") as f:
                    hdu = fits.open(f)
                    if len(hdu) > 1 and hdu[1].columns is not None:
                        wl_col = next((c for c in hdu[1].columns.names if "wave" in c.lower()), None)
                        fl_col = next(
                            (c for c in hdu[1].columns.names if c.lower() in ("flux", "spectrum", "signal")),
                            None,
                        )
                        if wl_col and fl_col:
                            wavelength = hdu[1].data[wl_col]
                            flux = hdu[1].data[fl_col]
                        else:
                            raise ValueError(f"Unexpected columns: {hdu[1].columns.names}")
                    else:
                        flux = hdu[0].data.ravel()
                        crval1 = hdu[0].header.get("CRVAL1", 0)
                        cdelt1 = hdu[0].header.get("CDELT1", 1)
                        crpix1 = hdu[0].header.get("CRPIX1", 1)
                        wavelength = crval1 + cdelt1 * (np.arange(len(flux)) - crpix1 + 1)
                ax.plot(wavelength, flux, label=f"{laser_wl} nm")
            except Exception as e:
                print(f"FiberSpectrograph {sal_idx} step {i} ({laser_wl} nm): {e}")

        ax.set_xlabel("Wavelength (nm)")
        ax.set_ylabel("Flux")
        ax.set_title(f"FiberSpectrograph {sal_idx}")
        ax.legend(fontsize=8, ncol=2)

    fig.suptitle(
        f"BLOCK-T683 FiberSpectrograph Spectra (laser_functional) – DayObs: {day_obs}", fontsize=12
    )
    plt.tight_layout()
    plt.show()
else:
    print("No FiberSpectrograph data to plot.")

## 3. TunableLaser Wavelength

The TunableLaser steps through multiple wavelengths during the `laser_functional` and `laser_cbp` sequences.

In [ ]:
laser_topic = "lsst.sal.TunableLaser.logevent_wavelength"
try:
    laser_df = await client.select_time_series(
        laser_topic, ["wavelength"], start=start_time, end=end_time
    )
    if len(laser_df) > 0:
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.step(laser_df.index, laser_df["wavelength"], where="post", linewidth=1.5)
        ax.set_xlabel("Time (UTC)")
        ax.set_ylabel("Wavelength (nm)")
        ax.set_title(f"TunableLaser Wavelength – DayObs: {day_obs}")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        print(f"Wavelengths visited: {sorted(laser_df['wavelength'].unique())}")
    else:
        print(f"No TunableLaser wavelength data for {day_obs}")
except Exception as e:
    print(f"Error querying TunableLaser wavelength: {e}")

## 4. CBP Motion Test

Verify that the CBP moved to each commanded position during the block. 
Dashed lines indicate the expected commanded values.

In [ ]:
# CBP azimuth and elevation – actual telemetry vs. commanded positions
# Expected: az 15.0° and -15.0°; el 0.0° and -20.0°
expected_az = [15.0, -15.0]
expected_el = [0.0, -20.0]

cbp_az_df = await client.select_time_series(
    "lsst.sal.CBP.azimuth", ["azimuth"], start=start_time, end=end_time
)
cbp_el_df = await client.select_time_series(
    "lsst.sal.CBP.elevation", ["elevation"], start=start_time, end=end_time
)
cbp_cmd_df = await client.select_time_series(
    "lsst.sal.CBP.command_move", ["azimuth", "elevation"], start=start_time, end=end_time
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

if len(cbp_az_df) > 0:
    ax1.plot(cbp_az_df.index, cbp_az_df["azimuth"], linewidth=1.5, label="Actual")
if len(cbp_cmd_df) > 0:
    ax1.scatter(cbp_cmd_df.index, cbp_cmd_df["azimuth"], marker="^", s=80,
                color="C3", zorder=5, label="Commanded")
for val in expected_az:
    ax1.axhline(val, linestyle="--", alpha=0.5, label=f"Expected: {val}°")
ax1.set_ylabel("Azimuth (°)")
ax1.set_title(f"CBP Azimuth – DayObs: {day_obs}")
ax1.legend()

if len(cbp_el_df) > 0:
    ax2.plot(cbp_el_df.index, cbp_el_df["elevation"], color="C1", linewidth=1.5, label="Actual")
if len(cbp_cmd_df) > 0:
    ax2.scatter(cbp_cmd_df.index, cbp_cmd_df["elevation"], marker="^", s=80,
                color="C3", zorder=5, label="Commanded")
for val in expected_el:
    ax2.axhline(val, linestyle="--", alpha=0.5, label=f"Expected: {val}°")
ax2.set_ylabel("Elevation (°)")
ax2.set_title(f"CBP Elevation – DayObs: {day_obs}")
ax2.legend()

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax2.set_xlabel("Time (UTC)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# CBP focus telemetry
# Expected: 5000 then 3800
expected_focus = [5000, 3800]

cbp_focus_df = await client.select_time_series(
    "lsst.sal.CBP.focus", ["focus"], start=start_time, end=end_time
)

fig, ax = plt.subplots(figsize=(13, 4))
if len(cbp_focus_df) > 0:
    ax.plot(cbp_focus_df.index, cbp_focus_df["focus"], linewidth=1.5, label="Actual")
    for val in expected_focus:
        ax.axhline(val, linestyle="--", alpha=0.5, label=f"Expected: {val}")
    ax.legend()
else:
    ax.text(0.5, 0.5, f"No CBP focus data for {day_obs}", transform=ax.transAxes, ha="center")
ax.set_title(f"CBP Focus – DayObs: {day_obs}")
ax.set_ylabel("Focus")
ax.set_xlabel("Time (UTC)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# CBP mask and mask rotation from status topic
# Expected masks: 5 then 1; rotations: 275.0° then 96.5°
expected_masks = [5, 1]
expected_rotations = [275.0, 96.5]

cbp_status_df = await client.select_time_series(
    "lsst.sal.CBP.status", ["mask", "mask_rotation"], start=start_time, end=end_time
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

if len(cbp_status_df) > 0:
    ax1.step(cbp_status_df.index, cbp_status_df["mask"], where="post", linewidth=2, label="Actual")
    for val in expected_masks:
        ax1.axhline(val, linestyle="--", alpha=0.5, label=f"Expected: {val}")
    ax1.set_ylabel("Mask")
    ax1.legend()
else:
    ax1.text(0.5, 0.5, f"No CBP status data for {day_obs}", transform=ax1.transAxes, ha="center")
ax1.set_title(f"CBP Mask – DayObs: {day_obs}")

if len(cbp_status_df) > 0:
    ax2.step(cbp_status_df.index, cbp_status_df["mask_rotation"], where="post",
             linewidth=2, color="C2", label="Actual")
    for val in expected_rotations:
        ax2.axhline(val, linestyle="--", alpha=0.5, label=f"Expected: {val}°")
    ax2.set_ylabel("Mask Rotation (°)")
    ax2.legend()
ax2.set_title(f"CBP Mask Rotation – DayObs: {day_obs}")

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax2.set_xlabel("Time (UTC)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Component Summary States

Verify that all calibration system components successfully transitioned:
**STANDBY → ENABLED** at the start and **ENABLED → STANDBY** at the end of the block.

SAL summary state values: `1`=DISABLED, `2`=ENABLED, `3`=FAULT, `4`=OFFLINE, `5`=STANDBY

In [ ]:
state_names = {1: "DISABLED", 2: "ENABLED", 3: "FAULT", 4: "OFFLINE", 5: "STANDBY"}

components = [
    ("LEDProjector", None),
    ("TunableLaser", None),
    ("Electrometer", 103),
    ("Electrometer", 102),
    ("LinearStage", 101),
    ("LinearStage", 102),
    ("LinearStage", 103),
    ("LinearStage", 104),
    ("CBP", None),
]

rows = []
for csc, sal_index in components:
    topic = f"lsst.sal.{csc}.logevent_summaryState"
    label = f"{csc}:{sal_index}" if sal_index else csc
    try:
        df = await client.select_time_series(
            topic, ["summaryState", "salIndex"], start=start_time, end=end_time
        )
        if sal_index is not None and "salIndex" in df.columns:
            df = df[df["salIndex"] == sal_index]
        if len(df) > 0:
            first_state = df["summaryState"].iloc[0]
            last_state = df["summaryState"].iloc[-1]
            states_seen = df["summaryState"].map(lambda x: state_names.get(x, str(x))).tolist()
            rows.append({
                "Component": label,
                "First State": state_names.get(first_state, first_state),
                "Last State": state_names.get(last_state, last_state),
                "All States": " → ".join(states_seen),
                "FAULT?": "YES" if 3 in df["summaryState"].values else "no",
            })
        else:
            rows.append({"Component": label, "First State": "no data", "Last State": "no data",
                         "All States": "", "FAULT?": "n/a"})
    except Exception as e:
        rows.append({"Component": label, "First State": f"error: {e}", "Last State": "",
                     "All States": "", "FAULT?": "n/a"})

state_table = pd.DataFrame(rows).set_index("Component")
state_table